In [3]:
"""
Classification Metrics and PDF Report Generator
===============================================

Usage:
    python classification_report.py --input data.csv
    python classification_report.py --input data.csv --output report.pdf

CSV Columns:
    Binary       : True labels (binary)
    Model_Durum  : Predicted labels (binary)
    Multi        : True labels (multi-class)
    Model_Tur    : Predicted labels (multi-class)
"""

import argparse
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, balanced_accuracy_score
)

PALETTE = sns.color_palette("Set2")
CM_CMAP  = "Blues"


# -------------------------------------------------
# Load Data
# -------------------------------------------------

def load_data(path: str) -> pd.DataFrame:

    df = pd.read_csv(path)

    required = {"Binary", "Model_Durum", "Multi", "Model_Tur"}

    missing = required - set(df.columns)

    if missing:
        sys.exit(f"[ERROR] Missing columns: {missing}")

    df = df.dropna(subset=list(required))

    print(f"[INFO] {len(df)} rows loaded.")

    return df



# -------------------------------------------------
# Metrics
# -------------------------------------------------
def fig_metrics_table(metrics: dict, title: str) -> plt.Figure:

    fig, ax = plt.subplots(
        figsize=(8,
        max(3,len(metrics)*0.45+1.5))
    )

    ax.set_axis_off()

    data=[
        [k,f"{v:.6f}"]
        for k,v in metrics.items()
    ]

    tbl=ax.table(
        cellText=data,
        colLabels=["Metric","Value"],
        cellLoc="center",
        loc="center",
        colColours=["#4472C4","#4472C4"]
    )

    tbl.auto_set_font_size(False)

    tbl.set_fontsize(11)

    tbl.scale(1.4,1.6)

    for (row,col),cell in tbl.get_celld().items():

        if row==0:

            cell.set_text_props(
                color="white",
                fontweight="bold"
            )

        elif row%2==0:

            cell.set_facecolor("#dce6f1")


    ax.set_title(
        title,
        fontweight="bold"
    )

    fig.tight_layout()

    return fig
def binary_metrics(y_true, y_pred) -> dict:

    metrics = {

        "Accuracy" :
            accuracy_score(y_true, y_pred),

        "Balanced Accuracy" :
            balanced_accuracy_score(y_true, y_pred),

        "Precision (macro)" :
            precision_score(y_true, y_pred,
                            average="macro",
                            zero_division=0),

        "Recall (macro)" :
            recall_score(y_true, y_pred,
                         average="macro",
                         zero_division=0),

        "F1 (macro)" :
            f1_score(y_true, y_pred,
                     average="macro",
                     zero_division=0),

        "F1 (weighted)" :
            f1_score(y_true, y_pred,
                     average="weighted",
                     zero_division=0),

        "Cohen Kappa" :
            cohen_kappa_score(y_true, y_pred),

        "MCC" :
            matthews_corrcoef(y_true, y_pred),
    }

    try:

        classes = np.unique(y_true)

        if len(classes) == 2:

            metrics["ROC-AUC"] = roc_auc_score(
                y_true,
                y_pred
            )

    except:
        pass

    return metrics



def multi_metrics(y_true, y_pred) -> dict:

    return {

        "Accuracy" :
            accuracy_score(y_true, y_pred),

        "Balanced Accuracy" :
            balanced_accuracy_score(y_true, y_pred),

        "Precision (macro)" :
            precision_score(y_true, y_pred,
                            average="macro",
                            zero_division=0),

        "Recall (macro)" :
            recall_score(y_true, y_pred,
                         average="macro",
                         zero_division=0),

        "F1 (macro)" :
            f1_score(y_true, y_pred,
                     average="macro",
                     zero_division=0),

        "F1 (weighted)" :
            f1_score(y_true, y_pred,
                     average="weighted",
                     zero_division=0),

        "Cohen Kappa" :
            cohen_kappa_score(y_true, y_pred),

        "MCC" :
            matthews_corrcoef(y_true, y_pred),
    }



# -------------------------------------------------
# Figures
# -------------------------------------------------

def fig_title_page(title: str, subtitle: str = "") -> plt.Figure:

    fig = plt.figure(figsize=(11.7, 8.3))

    fig.patch.set_facecolor("#1a1a2e")

    ax = fig.add_axes([0, 0, 1, 1])

    ax.set_axis_off()

    ax.text(
        0.5,
        0.58,
        title,
        fontsize=36,
        fontweight="bold",
        color="white",
        ha="center",
        va="center"
    )

    if subtitle:

        ax.text(
            0.5,
            0.44,
            subtitle,
            fontsize=16,
            color="#aaaacc",
            ha="center",
            va="center"
        )

    ax.text(
        0.5,
        0.15,
        "Classification Analysis Report",
        fontsize=13,
        color="#888899",
        ha="center",
        va="center"
    )

    return fig



# -------------------------------------------------

def fig_metrics_bar(metrics: dict, title: str) -> plt.Figure:

    labels = list(metrics.keys())
    values = list(metrics.values())

    colors = [
        PALETTE[i % len(PALETTE)]
        for i in range(len(labels))
    ]

    fig, ax = plt.subplots(figsize=(11.7, 5.5))

    bars = ax.barh(labels, values,
                   color=colors,
                   edgecolor="white",
                   linewidth=0.5)

    ax.set_xlim(0, 1.12)

    ax.set_xlabel("Score")

    ax.set_title(title,
                 fontweight="bold")

    ax.axvline(1.0,
               color="gray",
               linestyle="--")

    for bar, val in zip(bars, values):

        ax.text(
            val + 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}",
            va="center"
        )

    ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()

    return fig



# -------------------------------------------------

def fig_confusion_matrix(y_true, y_pred, labels, title: str) -> plt.Figure:

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    fig, axes = plt.subplots(1, 2,
                             figsize=(14, 5.5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap=CM_CMAP,
        xticklabels=labels,
        yticklabels=labels,
        ax=axes[0]
    )

    axes[0].set_title(title+" (Numeric)")

    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    axes[0].tick_params(axis='x',
                        rotation=90)



    cm_norm = cm.astype(float) / cm.sum(axis=1,
                                        keepdims=True)

    cm_norm = np.nan_to_num(cm_norm)


    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap=CM_CMAP,
        xticklabels=labels,
        yticklabels=labels,
        ax=axes[1],
        vmin=0,
        vmax=1
    )

    axes[1].set_title(title+" (Normalized)")

    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")

    axes[1].tick_params(axis='x',
                        rotation=90)

    fig.tight_layout(pad=2)

    return fig



# -------------------------------------------------

def fig_per_class_metrics(y_true, y_pred, labels, title: str) -> plt.Figure:

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0
    )

    rows = {
        str(lbl):
        report.get(str(lbl), {})
        for lbl in labels
    }

    df_rep = pd.DataFrame(rows).T[
        ["precision","recall","f1-score"]
    ].astype(float)


    fig, ax = plt.subplots(
        figsize=(11.7,
        max(4,len(labels)*0.6+2))
    )


    x = np.arange(len(df_rep))

    w = 0.25


    ax.bar(x-w,
           df_rep["precision"],
           width=w,
           label="Precision")

    ax.bar(x,
           df_rep["recall"],
           width=w,
           label="Recall")

    ax.bar(x+w,
           df_rep["f1-score"],
           width=w,
           label="F1-Score")


    ax.set_xticks(x)

    ax.set_xticklabels(
        df_rep.index,
        rotation=90
    )

    ax.set_ylim(0,1.15)

    ax.set_ylabel("Score")

    ax.set_title(
        title+" Per-Class Metrics",
        fontweight="bold"
    )

    ax.legend()

    fig.tight_layout()

    return fig



# -------------------------------------------------

def fig_class_distribution(y_true, y_pred, labels, title: str) -> plt.Figure:

    fig, axes = plt.subplots(1,2,
                             figsize=(12,4.5))


    counts_true = pd.Series(y_true)\
        .value_counts()\
        .reindex(labels,
                 fill_value=0)


    counts_pred = pd.Series(y_pred)\
        .value_counts()\
        .reindex(labels,
                 fill_value=0)



    axes[0].bar(
        counts_true.index.astype(str),
        counts_true.values
    )

    axes[0].set_title(
        title+" Actual Distribution"
    )

    axes[0].set_xlabel("Class")
    axes[0].set_ylabel("Count")

    axes[0].tick_params(
        axis='x',
        rotation=90
    )



    axes[1].bar(
        counts_pred.index.astype(str),
        counts_pred.values
    )

    axes[1].set_title(
        title+" Predicted Distribution"
    )

    axes[1].set_xlabel("Class")
    axes[1].set_ylabel("Count")

    axes[1].tick_params(
        axis='x',
        rotation=90
    )


    fig.tight_layout()

    return fig



# -------------------------------------------------
# Main
# -------------------------------------------------

def main():

    inputfile="veri.csv"

    outputfile="report.pdf"


    df = load_data(inputfile)


    y_bin_true = df["Binary"].values
    y_bin_pred = df["Model_Durum"].values

    y_mul_true = df["Multi"].values
    y_mul_pred = df["Model_Tur"].values


    bin_labels = sorted(
        np.unique(
        np.concatenate(
        [y_bin_true,
         y_bin_pred]))
    )


    mul_labels = sorted(
        np.unique(
        np.concatenate(
        [y_mul_true,
         y_mul_pred]))
    )


    bin_m = binary_metrics(
        y_bin_true,
        y_bin_pred
    )


    mul_m = multi_metrics(
        y_mul_true,
        y_mul_pred
    )



    print("\nBinary Metrics")

    for k,v in bin_m.items():

        print(f"{k:<22}:{v:.6f}")


    print("\nMulti Metrics")

    for k,v in mul_m.items():

        print(f"{k:<22}:{v:.6f}")



    with PdfPages(outputfile) as pdf:


        pdf.savefig(
            fig_title_page(
            "Classification Analysis Report",
            f"Source: {inputfile}")
        )

        plt.close("all")


        pdf.savefig(
            fig_metrics_table(
            bin_m,
            "Binary Metrics Table")
        )

        plt.close("all")


        pdf.savefig(
            fig_metrics_bar(
            bin_m,
            "Binary Metrics")
        )

        plt.close("all")


        pdf.savefig(
            fig_confusion_matrix(
            y_bin_true,
            y_bin_pred,
            bin_labels,
            "Binary Confusion Matrix")
        )

        plt.close("all")


        pdf.savefig(
            fig_per_class_metrics(
            y_bin_true,
            y_bin_pred,
            bin_labels,
            "Binary")
        )

        plt.close("all")


        pdf.savefig(
            fig_class_distribution(
            y_bin_true,
            y_bin_pred,
            bin_labels,
            "Binary")
        )

        plt.close("all")



        pdf.savefig(
            fig_metrics_table(
            mul_m,
            "Multi Metrics Table")
        )

        plt.close("all")


        pdf.savefig(
            fig_metrics_bar(
            mul_m,
            "Multi Metrics")
        )

        plt.close("all")


        pdf.savefig(
            fig_confusion_matrix(
            y_mul_true,
            y_mul_pred,
            mul_labels,
            "Multi Confusion Matrix")
        )

        plt.close("all")


        pdf.savefig(
            fig_per_class_metrics(
            y_mul_true,
            y_mul_pred,
            mul_labels,
            "Multi")
        )

        plt.close("all")


        pdf.savefig(
            fig_class_distribution(
            y_mul_true,
            y_mul_pred,
            mul_labels,
            "Multi")
        )

        plt.close("all")


    print("\nReport saved →",outputfile)



if __name__=="__main__":
    main()

[INFO] 1072 rows loaded.

Binary Metrics
Accuracy              :0.611007
Balanced Accuracy     :0.608264
Precision (macro)     :0.736930
Recall (macro)        :0.608264
F1 (macro)            :0.548453
F1 (weighted)         :0.549707
Cohen Kappa           :0.217713
MCC                   :0.320318
ROC-AUC               :0.608264

Multi Metrics
Accuracy              :0.780784
Balanced Accuracy     :0.785611
Precision (macro)     :0.820140
Recall (macro)        :0.785611
F1 (macro)            :0.782799
F1 (weighted)         :0.782427
Cohen Kappa           :0.726579
MCC                   :0.735770

Report saved → report.pdf
